In [ ]:

from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt


# ============================================================
# USER SETTINGS
# ============================================================

ROOT = Path.cwd()

MAE_DATA_DIR = ROOT / "Data" / "testing_data"

MAE_FILES = {
    4: MAE_DATA_DIR / "mae_values_n4.npy",
    6: MAE_DATA_DIR / "mae_values_n6.npy",
    8: MAE_DATA_DIR / "mae_values_n8.npy",
}

OUTPUT_DIR = ROOT / "Results" / "testing_mae_histograms"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:

VERSION = 8

SAVE_PNG = True
SAVE_PDF = True
SAVE_DPI = 500


# ============================================================
# PLOT CONTROL PARAMETERS
# ============================================================

X_SCALE = 1e6
X_SCALE_LABEL = r"$\times 10^{-6}$"

NUM_BINS = 35

FIGSIZE_COMBINED = (20, 8)

HIST_ALPHA = 0.85
HIST_EDGE_COLOR = "black"
HIST_LINEWIDTH = 0.7

MEAN_LINE_COLOR = "red"
MEAN_LINEWIDTH = 2.2

SHOW_GRID = False


# ============================================================
# FONT AND STYLE SETTINGS
# ============================================================

plt.rcParams.update({
    "font.size": 18,
    "axes.titlesize": 20,
    "axes.labelsize": 18,
    "xtick.labelsize": 16,
    "ytick.labelsize": 16,
    "legend.fontsize": 16,

    "lines.linewidth": 2.0,
    "axes.linewidth": 0.9,
    "xtick.major.width": 0.8,
    "ytick.major.width": 0.8,
    "xtick.major.size": 3.5,
    "ytick.major.size": 3.5,

    "figure.dpi": 150,
    "savefig.dpi": SAVE_DPI,

    "figure.facecolor": "white",
    "axes.facecolor": "white",

    "pdf.fonttype": 42,
    "ps.fonttype": 42,

    "axes.unicode_minus": False,
})


# ============================================================
# FONT SIZE CONTROLS
# ============================================================

LABEL_FONTSIZE = 22
TICK_FONTSIZE = 18
LEGEND_FONTSIZE = 22
MEAN_TEXT_FONTSIZE = 16

X_LABELPAD = 8
Y_LABELPAD = 8
TICK_PAD = 5


# ============================================================
# LAYOUT CONTROLS
# ============================================================

W_SPACE = 2.0

# Shared top legend position
LEGEND_BBOX = (0.5, 0.98)

# Leave top space for shared legend
LAYOUT_RECT = [0, 0, 1, 0.86]


# ============================================================
# HELPER FUNCTIONS
# ============================================================

def load_mae_values(path):
    """Load MAE values from .npy file."""
    path = Path(path)

    if not path.exists():
        raise FileNotFoundError(f"File not found:\n{path}")

    values = np.load(path)
    values = np.asarray(values, dtype=float).ravel()

    values = values[np.isfinite(values)]

    if len(values) == 0:
        raise ValueError(f"No valid MAE values found in:\n{path}")

    return values


def style_axis(ax):
    """Apply consistent paper-style axis formatting."""
    if SHOW_GRID:
        ax.grid(True, alpha=0.25, linewidth=0.6)
    else:
        ax.grid(False)

    ax.tick_params(
        axis="both",
        labelsize=TICK_FONTSIZE,
        width=0.8,
        length=3.5,
        direction="out",
        pad=TICK_PAD,
    )

    for spine in ax.spines.values():
        spine.set_linewidth(0.9)


def save_figure(fig, filename):
    """Save figure as PNG and/or PDF."""
    if SAVE_PNG:
        png_path = OUTPUT_DIR / f"{filename}.png"
        fig.savefig(png_path, dpi=SAVE_DPI, bbox_inches="tight")
        print(f"Saved PNG: {png_path.resolve()}")

    if SAVE_PDF:
        pdf_path = OUTPUT_DIR / f"{filename}.pdf"
        fig.savefig(pdf_path, bbox_inches="tight")
        print(f"Saved PDF: {pdf_path.resolve()}")

    plt.close(fig)


def print_mae_summary(n, mae_values):
    """Print numerical summary."""
    print("-" * 70)
    print(f"n = {n}")
    print(f"Number of samples : {len(mae_values)}")
    print(f"Mean MAE          : {np.mean(mae_values):.10e}")
    print(f"Min MAE           : {np.min(mae_values):.10e}")
    print(f"Max MAE           : {np.max(mae_values):.10e}")
    print(f"Std MAE           : {np.std(mae_values):.10e}")


# ============================================================
# MAIN COMBINED HISTOGRAM FUNCTION
# ============================================================

def plot_combined_histograms(mae_data):
    """
    Plot MAE histograms side-by-side.

    Correct behavior:
        - Shared top legend shows only MAE distribution.
        - Each subplot has a local mean legend with red line + mean value.
        - Red vertical line marks mean.
        - No subplot titles.
        - Only one y-axis label on the first subplot.
    """
    nodes = sorted(mae_data.keys())
    num_nodes = len(nodes)

    if num_nodes == 0:
        raise ValueError("No MAE data provided.")

    fig, axes = plt.subplots(
        1,
        num_nodes,
        figsize=FIGSIZE_COMBINED,
        sharey=False,
    )

    if num_nodes == 1:
        axes = [axes]

    for ax, n in zip(axes, nodes):
        mae_values = mae_data[n]

        mae_scaled = mae_values * X_SCALE

        mean_raw = np.mean(mae_values)
        mean_scaled = mean_raw * X_SCALE

        # Histogram
        hist = ax.hist(
            mae_scaled,
            bins=NUM_BINS,
            alpha=HIST_ALPHA,
            edgecolor=HIST_EDGE_COLOR,
            linewidth=HIST_LINEWIDTH,
            label="MAE distribution",
        )

        # Mean vertical line
        mean_line = ax.axvline(
            mean_scaled,
            color=MEAN_LINE_COLOR,
            linewidth=MEAN_LINEWIDTH,
        )

        # Local mean legend inside each subplot
        mean_legend = ax.legend(
            [mean_line],
            [f"Mean = {mean_raw:.2e}"],
            loc="upper right",
            fontsize=MEAN_TEXT_FONTSIZE,
            frameon=True,
            framealpha=0.92,
            edgecolor="0.75",
            handlelength=1.8,
            borderpad=0.3,
            labelspacing=0.2,
        )

        ax.add_artist(mean_legend)

        ax.set_xlabel(
            f"Sample wise MAE ({X_SCALE_LABEL})",
            fontsize=LABEL_FONTSIZE,
            labelpad=X_LABELPAD,
        )

        ax.set_title("")

        style_axis(ax)

    # One y-label only
    axes[0].set_ylabel(
        "Sample Count",
        fontsize=LABEL_FONTSIZE,
        labelpad=Y_LABELPAD,
    )

    for ax in axes[1:]:
        ax.set_ylabel("")

    # --------------------------------------------------------
    # Shared top legend: only MAE distribution
    # --------------------------------------------------------

    handles, labels = axes[0].get_legend_handles_labels()

    fig.legend(
        [handles[0]],
        ["MAE distribution"],
        loc="upper center",
        ncol=1,
        fontsize=LEGEND_FONTSIZE,
        frameon=True,
        framealpha=0.92,
        edgecolor="0.75",
        handlelength=1.8,
        borderpad=0.4,
        labelspacing=0.3,
        bbox_to_anchor=LEGEND_BBOX,
    )

    fig.tight_layout(
        pad=0.8,
        w_pad=W_SPACE,
        rect=LAYOUT_RECT,
    )

    save_figure(
        fig,
        f"testing_mae_histograms_combined_v{VERSION}",
    )


# ============================================================
# MAIN SCRIPT
# ============================================================

if __name__ == "__main__":

    mae_data = {}

    for n, path in MAE_FILES.items():
        print(f"\nLoading MAE values for n = {n}")
        print(f"Path: {path}")

        mae_values = load_mae_values(path)
        mae_data[n] = mae_values

        print_mae_summary(n, mae_values)

    plot_combined_histograms(mae_data)

    print("\nAll combined testing MAE histogram plots generated.")